# **Actividad 2**
### Rafael Santamaria Heredia | A01646667

In [10]:
import numpy as np
import pandas as pd

# Datos base
datos = pd.DataFrame({
    'Estudiante': range(1, 11),
    'Horas': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Nota': [3.2, 4.5, 5.0, 6.8, 7.0, 8.5, 9.0, 9.2, 9.8, 10.0]
})

x = datos['Horas'].to_numpy(dtype=float)
y = datos['Nota'].to_numpy(dtype=float)
n = len(x)

In [ ]:
# ==========================================
# 1. MODELO LINEAL POR MCO (FÓRMULA CERRADA)
# ==========================================
x_bar = np.mean(x)
y_bar = np.mean(y)

SS_xx = np.sum((x - x_bar) ** 2)
SS_yy = np.sum((y - y_bar) ** 2)
SS_xy = np.sum((x - x_bar) * (y - y_bar))

beta_1 = SS_xy / SS_xx
beta_0 = y_bar - beta_1 * x_bar

y_pred = beta_0 + beta_1 * x
residuals = y - y_pred

print("=== 1. MODELO LINEAL ===")
print(f"y_hat = {beta_0:.4f} + {beta_1:.4f} * x\n")

=== 1. MODELO LINEAL ===
y_hat = 3.0533 + 0.7721 * x



In [ ]:
# ==========================================
# 2. TABLA ANOVA
# ==========================================
SSR = np.sum((y_pred - y_bar) ** 2)
SSE = np.sum(residuals ** 2)
SST = SS_yy

df_reg = 1
df_res = n - 2
df_tot = n - 1

MSR = SSR / df_reg
MSE = SSE / df_res

tabla_anova = pd.DataFrame({
    'Fuente': ['Regresion', 'Error (Residual)', 'Total'],
    'SC': [SSR, SSE, SST],
    'gl': [df_reg, df_res, df_tot],
    'CM': [MSR, MSE, np.nan],
    'F0': [MSR / MSE, np.nan, np.nan]
})

print("=== 2. TABLA ANOVA ===")
print(tabla_anova.to_string(index=False), "\n")

=== 2. TABLA ANOVA ===
          Fuente        SC  gl        CM         F0
       Regresion 49.184121   1 49.184121 152.752906
Error (Residual)  2.575879   8  0.321985        NaN
           Total 51.760000   9       NaN        NaN 



In [ ]:
# ==========================================
# 3. PRUEBAS F Y t
# ==========================================
F_0 = MSR / MSE

# Errores estándar
se_beta_1 = np.sqrt(MSE / SS_xx)
se_beta_0 = np.sqrt(MSE * (1/n + (x_bar**2) / SS_xx))

t_beta_1 = beta_1 / se_beta_1
t_beta_0 = beta_0 / se_beta_0

# Valor crítico t de dos colas al 95% para gl = 8 (alpha = 0.05)
t_crit = 2.3060
F_crit = t_crit ** 2  # 5.3176

print("=== 3. PRUEBAS DE HIPÓTESIS ===")
print(f"Prueba F: F_0 = {F_0:.4f} | F_crit(0.05, 1, 8) = {F_crit:.4f}")
print(f"Prueba t (beta_1): t_0 = {t_beta_1:.4f} | t_crit(0.025, 8) = {t_crit:.4f}")
print(f"Prueba t (beta_0): t_0 = {t_beta_0:.4f} | t_crit(0.025, 8) = {t_crit:.4f}\n")

=== 3. PRUEBAS DE HIPÓTESIS ===
Prueba F: F_0 = 152.7529 | F_crit(0.05, 1, 8) = 5.3176
Prueba t (beta_1): t_0 = 12.3593 | t_crit(0.025, 8) = 2.3060
Prueba t (beta_0): t_0 = 7.8769 | t_crit(0.025, 8) = 2.3060



In [ ]:
# ==========================================
# 4. VERIFICACIÓN RELACIÓN F Y t
# ==========================================
print("=== 4. VERIFICACIÓN F == t^2 ===")
print(f"(t_0)^2 = {t_beta_1**2:.6f}")
print(f"F_0     = {F_0:.6f}")
print(f"Diferencia: {abs(F_0 - t_beta_1**2):.1e}\n")

=== 4. VERIFICACIÓN F == t^2 ===
(t_0)^2 = 152.752906
F_0     = 152.752906
Diferencia: 0.0e+00



In [ ]:
# ==========================================
# 5. INTERVALOS DE CONFIANZA PARA PARÁMETROS
# ==========================================
ci_beta_0 = (beta_0 - t_crit * se_beta_0, beta_0 + t_crit * se_beta_0)
ci_beta_1 = (beta_1 - t_crit * se_beta_1, beta_1 + t_crit * se_beta_1)

print("=== 5. INTERVALOS DE CONFIANZA AL 95% (PARÁMETROS) ===")
print(f"IC beta_0: [{ci_beta_0[0]:.4f}, {ci_beta_0[1]:.4f}]")
print(f"IC beta_1: [{ci_beta_1[0]:.4f}, {ci_beta_1[1]:.4f}]\n")

=== 5. INTERVALOS DE CONFIANZA AL 95% (PARÁMETROS) ===
IC beta_0: [2.1595, 3.9472]
IC beta_1: [0.6281, 0.9162]



In [ ]:
# ==========================================
# 6. TABLA IC DE LA MEDIA E INTERVALO DE PREDICCIÓN (IP)
# ==========================================
se_mean = np.sqrt(MSE * (1/n + ((x - x_bar)**2) / SS_xx))
se_pred = np.sqrt(MSE * (1 + 1/n + ((x - x_bar)**2) / SS_xx))

tabla_intervalos = pd.DataFrame({
    'Estudiante': datos['Estudiante'],
    'x': x,
    'y': y,
    'y_hat': y_pred,
    'IC_Media_Inf': y_pred - t_crit * se_mean,
    'IC_Media_Sup': y_pred + t_crit * se_mean,
    'IP_Inf': y_pred - t_crit * se_pred,
    'IP_Sup': y_pred + t_crit * se_pred
})

print("=== 6. TABLA DE IC DE LA MEDIA E IP (95%) ===")
print(tabla_intervalos.to_string(index=False))

=== 6. TABLA DE IC DE LA MEDIA E IP (95%) ===
 Estudiante    x    y     y_hat  IC_Media_Inf  IC_Media_Sup   IP_Inf    IP_Sup
          1  1.0  3.2  3.825455      3.056373      4.594536 2.307665  5.343244
          2  2.0  4.5  4.597576      3.945306      5.249845 3.135504  6.059647
          3  3.0  5.0  5.369697      4.821124      5.918269 3.950849  6.788545
          4  4.0  6.8  6.141818      5.675003      6.608633 4.752533  7.531104
          5  5.0  7.0  6.913939      6.493930      7.333949 5.539674  8.288205
          6  6.0  8.5  7.686061      7.266051      8.106070 6.311795  9.060326
          7  7.0  9.0  8.458182      7.991367      8.924997 7.068896  9.847467
          8  8.0  9.2  9.230303      8.681731      9.778876 7.811455 10.649151
          9  9.0  9.8 10.002424      9.350155     10.654694 8.540353 11.464496
         10 10.0 10.0 10.774545     10.005464     11.543627 9.256756 12.292335


In [9]:
from google.colab import drive, files
import os

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Pide la ruta completa del notebook (o puedes pegarla directamente en la variable)
ruta_ipynb = input("Pega aquí la ruta del archivo .ipynb en tu Drive: ").strip().replace("'", "").replace('"', '')

# Validar y convertir
if os.path.exists(ruta_ipynb):
    # Convertir a HTML
    os.system(f"jupyter nbconvert --to html '{ruta_ipynb}'")

    # Ruta del archivo HTML generado (se crea en la misma carpeta del ipynb)
    ruta_html = os.path.splitext(ruta_ipynb)[0] + ".html"

    # Descargar a tu computadora
    files.download(ruta_html)
    print("✅ Descarga iniciada con éxito.")
else:
    print(f"❌ No se encontró el archivo en: {ruta_ipynb}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pega aquí la ruta del archivo .ipynb en tu Drive: /content/drive/MyDrive/Aplicación de métodos multivariados en ciencia de datos/ACTIVIDAD 2.ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Descarga iniciada con éxito.
